In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples

In [2]:
n = pypsa.Network()
n.import_from_netcdf('C:/Users/jy/OneDrive - TransitionZero/tza-pypsa/reference_run_30-04-2025_update_with_2022_demand_profile.nc')

INFO:pypsa.io:Imported network reference_run_30-04-2025_update_with_2022_demand_profile.nc has buses, carriers, generators, links, loads, storage_units


In [23]:
generator_lookup = (n
                    .generators
                    .reset_index()[['Generator', 'bus', 'type']]
                    .rename(
                        columns={'bus': 'Node', 'type': 'Tech'})
                        )

In [25]:
storage_lookup = (n
                  .storage_units
                  .reset_index()[['StorageUnit', 'bus', 'type']]
                  .rename(
                      columns={'bus': 'Node', 'type': 'Tech'})
                      )

In [26]:
interconnector_lookup = (n
                         .links
                         .reset_index()[['Link', 'bus0', 'bus1']]
                            .rename(
                                columns={'bus0': 'Node', 'bus1': 'Node_Destination'})
                                )

In [42]:
generation = (
    n
    .generators_t
    .p
    .reset_index()
)

storage = (
    n
    .storage_units_t
    .p
    .reset_index()
)

interconnector_p0 = (
    n
    .links_t
    .p0
    .reset_index()
)

interconnector_p1 = (
    n
    .links_t
    .p1
    .reset_index()
)

In [43]:
generation = pd.melt(
    generation,
    id_vars='snapshot',
    var_name='Generator', 
    value_name='Value'
)

storage = pd.melt(
    storage,
    id_vars='snapshot',
    var_name='StorageUnit', 
    value_name='Value'
)

interconnector_p0 = pd.melt(
    interconnector_p0,
    id_vars='snapshot',
    var_name='Link', 
    value_name='Value'
)

interconnector_p1 = pd.melt(
    interconnector_p1,
    id_vars='snapshot',
    var_name='Link', 
    value_name='Value'
)

In [30]:
generation = pd.merge(
    generation,
    generator_lookup,
    on='Generator',
    how='left'
)

In [39]:
storage = pd.merge(
    storage,
    storage_lookup,
    on='StorageUnit',
    how='left'
)

In [48]:
interconnector_p0 = pd.merge(
    interconnector_p0,
    interconnector_lookup,
    on='Link',
    how='left'
)

In [50]:
interconnector_p1 = pd.merge(
    interconnector_p1,
    interconnector_lookup,
    on='Link',
    how='left'
).rename(
    columns={'Node': 'Node_Destination', 'Node_Destination': 'Node'}
)

In [51]:
interconnector_p1

,snapshot,Link,Value,Node_Destination,Node
0,2030-01-01 00:00:00,JPN01-JPN02-2030,-970.000000,JPN01,JPN02
1,2030-01-01 01:00:00,JPN01-JPN02-2030,-970.000000,JPN01,JPN02
2,2030-01-01 02:00:00,JPN01-JPN02-2030,-970.000000,JPN01,JPN02
3,2030-01-01 03:00:00,JPN01-JPN02-2030,-970.000000,JPN01,JPN02
4,2030-01-01 04:00:00,JPN01-JPN02-2030,-970.000000,JPN01,JPN02
...,...,...,...,...,...
175195,2030-12-31 19:00:00,JPN09-JPN07-2030,-0.000000,JPN09,JPN07
175196,2030-12-31 20:00:00,JPN09-JPN07-2030,-0.000000,JPN09,JPN07
175197,2030-12-31 21:00:00,JPN09-JPN07-2030,-641.497711,JPN09,JPN07
175198,2030-12-31 22:00:00,JPN09-JPN07-2030,-0.000000,JPN09,JPN07
